In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig, VitsModel, TextStreamer
import time
import psutil
import torch
import whisper
import sounddevice as sd
import numpy as np
from IPython.display import Audio
from queue import Queue
import threading
import numpy as np


c:\Users\Alysha\Documents\kata-ondevice\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- LLAMA ---

model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
quantization_config = QuantoConfig(weights="int8")

model = AutoModelForCausalLM.from_pretrained(model_id,
    device_map="cpu",
    quantization_config=quantization_config)

In [21]:
# --- STT ---

whisper_model = whisper.load_model("small")

def perintah():
    duration = 5
    sample_rate = 16000  

    print("Mendengarkan......")
    audio_data = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
    sd.wait()  
    print("Diterima.....")

    audio_data = np.squeeze(audio_data)  
    dengar = whisper_model.transcribe(audio_data, fp16=False, language="id")

    return dengar["text"], audio_data

In [4]:
# --- TTS ---

mms = VitsModel.from_pretrained("facebook/mms-tts-ind")
mms_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ind")

# def ngomong(text):
#     inputs = mms_token(text, return_tensors="pt")

#     with torch.no_grad():
#         output = mms(**inputs).waveform
#         return Audio(output.squeeze().cpu().numpy(), rate=16000) # Can change rate to make it faster/slower

def ngomong(text: str):
    """
    Converts text to speech using the MMS TTS model (facebook/mms-tts-ind) and plays the audio.
    Args:
        text (str): The input text to be synthesized into speech.
    """
    inputs = mms_tokenizer(text, return_tensors="pt")
    
    with torch.no_grad():
        output = mms(**inputs).waveform
    
    audio_array = output.squeeze().cpu().numpy()
    
    sample_rate = 16000  # You can adjust this based on the model's specifications if necessary
    
    sd.play(audio_array, sample_rate)
    sd.wait() 


In [25]:
def get_llm_response(text: str) -> str:
    """
    Generates a response to the given text using the Llama-2 language model.
    Args:
        text (str): The input text to be processed.
    Returns:
        str: The generated response.
    """
    # Prepare the prompt
    prompt = f"User: {text}\nAssistant: Tolong jawab singkat kurang dari 20 kata."

    # Tokenize input and generate response
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        inputs.input_ids,
        max_length=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract assistant's response
    response = response.split("Assistant:")[-1].strip()
    return response

In [27]:
def main_loop():
    """Main loop for the app"""
    try:
        while True:
            input("Press Enter to start recording.")

            # Process the recorded audio and transcription
            text, audio_data = perintah()

            if audio_data.size > 0:
                print(f"You: {text}")
                print("Generating response...")
                
                response = get_llm_response(text)
                print(f"Assistant: {response}")
                
                ngomong(response) 
            else:
                print("No audio recorded. Please ensure your microphone is working.")
                
    except KeyboardInterrupt:
        print("\nExiting...")

In [ ]:
if __name__ == "__main__":
    main_loop()

Mendengarkan......
Diterima.....


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


You:  Halo apa kabar?
Generating response...
Assistant: Tolong jawab singkat kurang dari 20 kata. Saya senang Anda bertemu!

Halo! Saya senang juga! Berapa kabar? Saya berada di tempat yang tenang dan aman. Apa kabar Anda?

Exiting...
